# Quantum Computing 

DOCDB families of the *quantum computing* sector on PATSTAT (EPO TIP), with search concepts from the file
`epo_patent_insight_report-quantum_computing-supplementary-materials_en.xlsx` (*Search concepts* sheet).

**Sector** = Core CPC (`G06N10/low`, `G06N99/002`) ∪ general keywords ∪ "physical implementations" keywords
(∪ broad keywords, if `INCLUDE_BROAD_TERMS = True`). The searched text is title + abstract.

In [2]:
from pathlib import Path

import pandas as pd
from sqlalchemy import and_, func, or_, select

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

ENV = "PROD"                 # rerun with "PROD" for the final result
START_YEAR, END_YEAR = 1990, 2023   # earliest filing year (earliest_filing_year)
IPR_TYPE = "PI"              # Patented Inventions: excludes utility models
INCLUDE_BROAD_TERMS = True   # quantum simulat+ / network+ / repeater

OUTPUT_DIR = Path("outputs/step0_epo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()
DIALECT = db.get_bind().dialect.name

## 1. Translation of EPO syntax into regular expressions

Text is converted to lowercase; "word" = sequence of `[a-z0-9]`, everything else is a separator.

| EPO Syntax | Meaning | Regex Translation |
|---|---|---|
| `a w b` | adjacent words, in order | `w(a, b)` |
| `a nw b` | in order, up to n words in between | `nw(a, b, n)` |
| `a d b`, `a nD b` | any order, up to n words in between (`d` = 0) | `nd(a, b, n)` |
| `+` | truncation, 0 or more characters | `T` = `[a-z0-9]*` |
| `?` | 0 or 1 character | `Q` = `[a-z0-9]?` |
| `_` | attached or separated | `J` = `[ -]?` |
| `[b,d,tr]` | string alternative | `alt(...)` = `(b|d|tr)` |

Each pattern is enclosed between word delimiters `(^|[^a-z0-9]) … ([^a-z0-9]|$)`; only constructs
common to PostgreSQL, BigQuery, Spark, and MySQL are used.

In [3]:
SEP = r"[^a-z0-9]+"   # separator between words
WORD = r"[a-z0-9]+"   # a word
T = r"[a-z0-9]*"      # EPO '+'
Q = r"[a-z0-9]?"      # EPO '?'
J = r"[ -]?"          # EPO '_'


def alt(*parts):
    """Alternative: (p1|p2|...)."""
    return "(" + "|".join(parts) + ")"


def w(*parts):
    """EPO 'w': adjacent words in the given order."""
    return "(" + SEP.join(parts) + ")"


def nw(a, b, n):
    """EPO 'nw': a then b, with at most n words in between."""
    return "(" + a + SEP + "(" + WORD + SEP + "){0," + str(n) + "}" + b + ")"


def nd(a, b, n):
    """EPO 'nD': a and b in any order, with at most n words in between."""
    return alt(nw(a, b, n), nw(b, a, n))


def word(p):
    """Delimits the pattern to whole words."""
    return "(^|[^a-z0-9])" + p + "([^a-z0-9]|$)"

## 2. Search concepts (from *Search concepts* sheet)

In [4]:
# --- Reused blocks ---------------------------------------------------------
QU_BDT = "(multi" + J + ")?qu" + J + "(b|d|tr)it" + Q          # [,multi]_qu_[b,d,tr]it?
QUBIT = alt(w("quantum", "bit" + Q), QU_BDT)                      # quantum w bit?, ...

QC_CORE = alt(                                                    # right-hand side of EPO '5d'
    QUBIT,
    w("quantum", "comput" + T),
    w("quantum", "gate" + Q),
    w("quantum", "circuit" + Q),
    w("quantum", "simulat" + T),
    w("quantum", "processor" + Q),
    w("quantum", "chip" + Q),
)

PHYS_TECH = alt(                                                  # left-hand side of EPO '5d'
    "wave" + J + "guide" + Q,
    "cavit(y|ies)",
    "resonator" + Q,
    "majorana",
    "cooper" + J + "pair" + Q,
    w("josephson", "junction" + Q),
    w(alt("nitrogen", "nv"), "vacan" + T),
    w("neutral", "atom" + Q),
    w("rydberg", alt("atom" + Q, "state" + Q)),
    "divincenzo",
    "kane",
    "quantum" + J + "dot" + Q,
    "super" + J + "conduct" + T,
    "squid" + Q,
    w("paul", "trap" + Q),
    nd("trap" + T, alt("ion" + Q, "atom" + Q), 0),                # trap+ d (ion?,atom?)
)

# --- Quantum computing in general -------------------------------------------
GENERAL_KW = {
    "instruction_code": w("quantum", alt("instruction" + Q, "(op" + J + ")?code" + Q)),
    "dev_kit": w("quantum", "develop" + T, "kit" + Q),
    "sdk": w("quantum", "sdk" + Q),
    "software": w("quantum", "software" + Q),
    "language": w("quantum", "lang" + T),
    "assembler": w("quantum", alt("assembler" + Q, w("assembly", "lang" + T))),
    "qiskit": "qiskit",
    "programming": w("quantum", "program" + T),
    "kernel": w("quantum", "kernel" + Q),
    "neural_net": w("quantum", "neur" + T, "net" + T),
    "machine_learning": w("quantum", "machine", "learning"),
    "qaoa": alt(
        w("quantum", "approximat" + T, "optimi" + T, "algo" + T),
        w("quantum", "alternat" + T, "operator" + Q, "ansatz"),
    ),
    "vqe": w("variational", "quantum", "eigen" + J + "solv" + T),
    "hadamard_fourier": w("quantum", alt("hadamard", "fourier")),
    "named_algorithms": nd(alt("deutsch" + J + "jozsa", "shor", "grover") + "s?", "algo" + T, 2),
    "turing_state_machine": w("quantum", alt("turing", w("state", "machine" + Q))),
    "topological_adiabatic": alt(w("topological", "quantum"), w("adiabatic", "quantum")),
    "named_gates": alt(w(alt("clifford", "hadamard", "swap", "pauli"), "gate" + Q), "t" + J + "gate" + Q),
    "annealing": w("quantum", "anneal" + T),
    "synthesis": w("quantum", "synthesis"),
    "compiler": w("quantum", "compil" + T),
    "qram": alt(w("quantum", alt("ram" + Q, w("random", "access", "memor" + T))), "q" + J + "ram" + Q),
    "information_processing": w("quantum", "information", "processing"),
    "chip_processor": w("quantum", alt(
        "chip" + Q,
        "processor" + Q,
        w("processing", alt("unit" + Q, "device" + Q, "apparatus" + Q + Q, "system" + Q)),
    )),
    "register": w("quantum", "register" + Q),
    "bus": w("quantum", "bus(es)?"),
    "fluxmon_transmon": alt("flux" + J + "mon" + Q, "trans" + J + "mon" + Q),
    "circuit": w("quantum", "circuit" + Q),
    "gate": w("quantum", "gate" + Q),
    "computing": w("quantum", "comput" + T),
    "qubit": QUBIT,
}

GENERAL_KW_BROAD = {
    "simulation": w("quantum", "simulat" + T),
    "repeater": w("quantum", "repeater" + Q),
    "network": w("quantum", "network" + T),
}

# --- Physical realisations of quantum computing ------------------------------
PHYS_KW = {
    "coupler": w(alt("quantum", QUBIT), "coupler" + Q),
    "qubit_control": nd(QUBIT, "control" + T, 2),
    "nisq": alt("nisq", w("noisy", "intermed" + T, "scale" + T, "quantum" + T)),
    "tech_near_qc": nd(PHYS_TECH, QC_CORE, 5),
    "qubit_type": w(alt("photon" + T, "flux" + T, "charge" + T, "phase" + T), QUBIT),
}

SECTOR_KW = {**GENERAL_KW, **PHYS_KW, **(GENERAL_KW_BROAD if INCLUDE_BROAD_TERMS else {})}

# --- Core CPC classes ('/low' translated into prefixes) ----------------------------
CPC_CORE = ("G06N10/", "G06N99/002")

# --- Economic prefilter (LIKE) before regex -----------------------------
ANCHORS = (
    "quantum", "qubit", "qudit", "qutrit", "qu bit", "qu-bit", "qiskit",
    "transmon", "trans mon", "trans-mon", "fluxmon", "flux mon", "flux-mon",
    "nisq", "shor", "grover", "deutsch", "clifford", "pauli", "hadamard",
    "swap gate", "t gate", "t-gate", "tgate", "qram", "q-ram", "q ram",
)

## 3. Query functions
`rx()` chooses the regex operator based on the backend SQL dialect.

In [5]:
def rx(col, pattern):
    """Boolean condition 'col contains pattern' in the backend SQL dialect."""
    d = DIALECT.lower()
    if d.startswith("postgres"):
        return col.op("~", is_comparison=True)(pattern)
    if d == "bigquery":
        return func.regexp_contains(col, pattern)
    if d in ("databricks", "spark", "sparksql", "hive"):
        return col.op("rlike", is_comparison=True)(pattern)
    if d in ("mysql", "mariadb"):
        return col.op("regexp", is_comparison=True)(pattern)
    return col.regexp_match(pattern)


def cpc_like(prefixes):
    symbol = func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "")
    return or_(*(symbol.like(p.replace(" ", "") + "%") for p in prefixes))


TXT = func.lower(func.concat(
    func.coalesce(TLS202_APPLN_TITLE.appln_title, ""),
    " ",
    func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, ""),
))

## 4. Query: sector families
Single query: application with core CPC **or** with at least one text match (`LIKE` prefilter + regex).

In [6]:
cpc_cond = TLS201_APPLN.appln_id.in_(
    select(TLS224_APPLN_CPC.appln_id).where(cpc_like(CPC_CORE))
)

text_cond = and_(
    or_(*(TXT.like(f"%{a}%") for a in ANCHORS)),
    or_(*(rx(TXT, word(p)) for p in SECTOR_KW.values())),
)

q = (
    db.query(TLS201_APPLN.docdb_family_id)
    .select_from(TLS201_APPLN)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(TLS201_APPLN.docdb_family_id > 0)
    .filter(TLS201_APPLN.earliest_filing_year.between(START_YEAR, END_YEAR))
    .filter(TLS201_APPLN.ipr_type == IPR_TYPE)
    .filter(or_(cpc_cond, text_cond))
    .distinct()
)

sector_ids = {row[0] for row in q.all()}
print("Families in sector:", len(sector_ids))

pd.DataFrame(sorted(sector_ids), columns=["docdb_family_id"]).to_csv(
    OUTPUT_DIR / f"qc_families_v2_{ENV}.csv", index=False
)

Families in sector: 12272


## 5. Comparison with v1 query


In [7]:
from pathlib import Path
import pandas as pd
from sqlalchemy import and_, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

ENV = "PROD"  # rerun with "PROD" for the final result
OUTPUT_DIR = Path("outputs/step0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()

SECTOR_NAME = "Quantum Computing"

# "Core" CPC classes: prefix without spaces
SECTOR_CPC_PREFIXES = (
    "G06N10/",      # Quantum computing (since 2019), includes /20 /40 /60 /70 /80
    "G06N99/002",   # Quantum computers (historical classification, pre-2019)
    "Y10S977/933",  # Spintronics or quantum computing (historical collection)
)

# "Adjacent" CPC classes: valid only if accompanied by quantum text
ADJACENT_CPC_PREFIXES = (
    "B82Y10/00",    # Nanotechnology for information processing
    "H10N60/",      # Superconducting devices
    "H01L39/",      # Superconducting devices (old classification)
    "H03K19/195",   # Superconducting logic
)

SECTOR_KEYWORDS = (
    "quantum comput",   # computer, computing, computation
    "qubit",
    "quantum bit",
    "quantum processor",
    "quantum gate",
    "quantum circuit",
    "quantum annealing",
    "quantum error correction",
    "quantum algorithm",
    "quantum simulator",
)

# Generic keywords: used ONLY in AND with adjacent CPC classes
ADJACENT_KEYWORDS = ("quantum", "josephson")
# The Josephson junction is the physical component on which superconducting qubits are based, which are the most widespread hardware platform in quantum computing
# They consist of two superconducting electrodes separated by an extremely thin insulating barrier, typically aluminum, about 1–2 nm thick.

START_YEAR = 1990   # earliest filing year of the family
END_YEAR = 2023

IPR_TYPE = "PI"  # "PI" = Patented Inventions (excludes utility models)


def symbol_prefix_condition(symbol_column, codes):
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


def keyword_condition(terms):
    title = func.lower(func.coalesce(TLS202_APPLN_TITLE.appln_title, ""))
    abstract = func.lower(func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, ""))
    return or_(*(col.like(f"%{t.lower()}%") for t in terms for col in (title, abstract)))


def sector_family_ids():
    family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")

    def base():
        return (
            db.query(family_id)
            .select_from(TLS201_APPLN)
            .filter(TLS201_APPLN.docdb_family_id > 0)  # excludes applications without family
            .filter(TLS201_APPLN.earliest_filing_year.between(START_YEAR, END_YEAR))
            .filter(TLS201_APPLN.ipr_type == IPR_TYPE) 
        )

    def join_cpc(q):
        return q.join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)

    def join_text(q):
        return (
            q.outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
            .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        )

    # Core CPC perimeter
    by_cpc = join_cpc(base()).filter(
        symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, SECTOR_CPC_PREFIXES)
    )

    # Specific keyword perimeter
    by_keywords = join_text(base()).filter(keyword_condition(SECTOR_KEYWORDS))

    # Adjacent CPC AND generic keyword
    by_adjacent = join_text(join_cpc(base())).filter(
        and_(
            symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, ADJACENT_CPC_PREFIXES),
            keyword_condition(ADJACENT_KEYWORDS),
        )
    )

    return by_cpc.union(by_keywords).union(by_adjacent)


sector_v1 = sector_family_ids()

In [8]:

v1 = {row[0] for row in set(sector_v1)}
v2 = sector_ids
print(f"v1: {len(v1)}  v2: {len(v2)}  comuni: {len(v1 & v2)}  solo v1: {len(v1 - v2)}  solo v2: {len(v2 - v1)}")
pd.Series(sorted(v1 - v2), name="docdb_family_id").to_csv(OUTPUT_DIR / "solo_v1.csv", index=False)
pd.Series(sorted(v2 - v1), name="docdb_family_id").to_csv(OUTPUT_DIR / "solo_v2.csv", index=False)


v1: 13620  v2: 12272  comuni: 11386  solo v1: 2234  solo v2: 886


## 6. Validation sample for LLM labelling
 Paste these cells at the end of the notebook (after section 5). They reuse `db`, `v1`, `v2`,
 `TXT`, the table models, START_YEAR, END_YEAR and IPR_TYPE already defined there.
 Run with ENV = "PROD": the TEST database has too little text to be useful.

In [13]:
import json
import random
from pathlib import Path
 
import pandas as pd
 
SEED = 42
N_PER_STRATUM = {"both": 50, "only_v1": 50, "only_v2": 50, "near_miss": 30}
OVERSAMPLE = 1.5          # draw extra families, because some have no English abstract
CHUNK = 500               # size of the IN (...) lists sent to the database
 
VAL_DIR = Path("outputs/step0_validation")
VAL_DIR.mkdir(parents=True, exist_ok=True)

### 6.1 Near-miss population
 Families that mention "quantum" in title/abstract but are in neither v1 nor v2.
 They are the only way to estimate what the query misses.

In [10]:
near_q = (
    db.query(TLS201_APPLN.docdb_family_id)
    .select_from(TLS201_APPLN)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(TLS201_APPLN.docdb_family_id > 0)
    .filter(TLS201_APPLN.earliest_filing_year.between(START_YEAR, END_YEAR))
    .filter(TLS201_APPLN.ipr_type == IPR_TYPE)
    .filter(TXT.like("%quantum%"))
    .distinct()
)
near_pool = {r[0] for r in near_q.all()} - (v1 | v2)
 
strata = {
    "both": v1 & v2,
    "only_v1": v1 - v2,
    "only_v2": v2 - v1,
    "near_miss": near_pool,
}
sizes = {k: len(s) for k, s in strata.items()}
print("Population sizes:", sizes)
(VAL_DIR / "strata_sizes.json").write_text(json.dumps(sizes, indent=2))

Population sizes: {'both': 11386, 'only_v1': 2234, 'only_v2': 886, 'near_miss': 65132}


78

### 6.2 Text of one application per family (English abstract)

In [11]:
def fetch_text(family_ids):
    """One row per family: the earliest application that has an English abstract."""
    ids, rows = list(family_ids), []
    for i in range(0, len(ids), CHUNK):
        rows += (
            db.query(
                TLS201_APPLN.docdb_family_id,
                TLS201_APPLN.appln_id,
                TLS202_APPLN_TITLE.appln_title,
                TLS203_APPLN_ABSTR.appln_abstract,
            )
            .select_from(TLS201_APPLN)
            .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
            .join(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
            .filter(TLS201_APPLN.docdb_family_id.in_(ids[i:i + CHUNK]))
            .filter(TLS201_APPLN.ipr_type == IPR_TYPE)
            .filter(TLS203_APPLN_ABSTR.appln_abstract_lg == "en")
            .all()
        )
    df = pd.DataFrame(rows, columns=["docdb_family_id", "appln_id", "title", "abstract"])
    df = df[df["abstract"].fillna("").str.len() > 50]
    return (df.sort_values("appln_id")
              .drop_duplicates("docdb_family_id")
              .set_index("docdb_family_id"))
 
 
rng = random.Random(SEED)
picked = []
for stratum, n in N_PER_STRATUM.items():
    pool = sorted(strata[stratum])
    rng.shuffle(pool)
    text = fetch_text(pool[: int(n * OVERSAMPLE)])
    keep = [f for f in pool[: int(n * OVERSAMPLE)] if f in text.index][:n]
    if len(keep) < n:
        print(f"WARNING: stratum {stratum}: only {len(keep)}/{n} families with an English abstract")
    for f in keep:
        picked.append((stratum, f, text.loc[f, "title"], text.loc[f, "abstract"]))
 
sample = pd.DataFrame(picked, columns=["stratum", "docdb_family_id", "title", "abstract"])
 


 ### 6.3 Export: one file for the LLMs, one private key

In [12]:
sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)   # shuffle strata together
sample["id"] = [f"S{i + 1:03d}" for i in range(len(sample))]
sample["in_v1"] = sample["docdb_family_id"].isin(v1).astype(int)
sample["in_v2"] = sample["docdb_family_id"].isin(v2).astype(int)
 
# LLMs see ONLY id, title, abstract (no CPC, no stratum, no family id)
sample[["id", "title", "abstract"]].to_csv(VAL_DIR / "to_label.csv", index=False)
# Private key: never shared with the LLMs
sample[["id", "docdb_family_id", "stratum", "in_v1", "in_v2"]].to_csv(
    VAL_DIR / "key_private.csv", index=False
)
print(sample["stratum"].value_counts())

stratum
both         50
only_v1      50
only_v2      50
near_miss    30
Name: count, dtype: int64
